# 04 — Simulación Monte Carlo del Mundial 2026

Este notebook ejecuta la simulación completa del Mundial FIFA 2026 con:

1. **Grupos oficiales** del sorteo FIFA 2026 (48 selecciones)
2. **Simulación Monte Carlo** con 100,000 iteraciones
3. **Intervalos de confianza** al 95% (Wilson Score + Bootstrap)
4. **Predicción de clasificados** por grupo (2 por grupo)
5. **Validación retrospectiva** con mundiales 2018 y 2022
6. **Visualizaciones** con barras de error

In [ ]:
import importlib
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_processed_dataset
from src.features.feature_engineering import build_feature_matrix
from src.simulation import monte_carlo as mc
from src.simulation.confidence import compute_all_confidence_intervals
from src.simulation.groups_2026 import GROUPS_2026, TEAMS_2026
from src.simulation.retrospective import (
    run_retrospective_validation,
    format_retrospective_report,
)

importlib.reload(mc)

run_fixed_group_tournament = mc.run_fixed_group_tournament

MODEL_PATH = PROJECT_ROOT / "models" / "trained" / "xgb.pkl"
N_SIMULATIONS = 100_000

print(f"Modelo: {MODEL_PATH}")
print(f"Simulaciones: {N_SIMULATIONS:,}")
print(f"Equipos: {len(TEAMS_2026)}")
print(f"Grupos: {len(GROUPS_2026)}")

## 1. Grupos oficiales del Mundial 2026

In [ ]:
# Mostrar los 12 grupos oficiales
for group_label, teams in GROUPS_2026.items():
    print(f"Grupo {group_label}: {', '.join(teams)}")

print(f"\nTotal de selecciones: {len(TEAMS_2026)}")

## 2. Construcción de probabilidades de partido

In [ ]:
# Preparar probabilidades con el modelo entrenado
import itertools

hist_df = load_processed_dataset()
features_df, feature_cols = build_feature_matrix(
    hist_df, include_elo=True, apply_decay=True, form_window=5
)

# Última fila por equipo como local y visitante
home_latest = (
    features_df.sort_values("date")
    .groupby("home_team", as_index=False)
    .tail(1)
    .set_index("home_team")
)
away_latest = (
    features_df.sort_values("date")
    .groupby("away_team", as_index=False)
    .tail(1)
    .set_index("away_team")
)

# Crear todas las combinaciones de partidos entre las 48 selecciones
pairs = [(h, a) for h, a in itertools.product(TEAMS_2026, TEAMS_2026) if h != a]
proba_base = pd.DataFrame(pairs, columns=["home_team", "away_team"])

# Crear features del fixture a partir de snapshots
fixture_features = proba_base.copy()

fixture_features = fixture_features.join(
    home_latest.add_suffix("_home"), on="home_team"
).join(
    away_latest.add_suffix("_away"), on="away_team"
)

# Reconstruir columnas diff esperadas
fixture_features["rank_diff"] = fixture_features["rank_home_home"] - fixture_features["rank_away_away"]
fixture_features["elo_diff"] = fixture_features["elo_home_home"] - fixture_features["elo_away_away"]
fixture_features["market_value_diff"] = fixture_features["market_value_diff_home"].fillna(0) - fixture_features["market_value_diff_away"].fillna(0)
fixture_features["avg_age_diff"] = fixture_features["avg_age_diff_home"].fillna(0) - fixture_features["avg_age_diff_away"].fillna(0)
fixture_features["squad_size_diff"] = fixture_features["squad_size_diff_home"].fillna(0) - fixture_features["squad_size_diff_away"].fillna(0)
fixture_features["top5_players_diff"] = fixture_features["top5_players_diff_home"].fillna(0) - fixture_features["top5_players_diff_away"].fillna(0)
fixture_features["home_advantage"] = 0  # Mundial es cancha neutral
fixture_features["form_win_rate_diff"] = fixture_features["form_win_rate_diff_home"].fillna(0) - fixture_features["form_win_rate_diff_away"].fillna(0)
fixture_features["form_goal_diff_diff"] = fixture_features["form_goal_diff_diff_home"].fillna(0) - fixture_features["form_goal_diff_diff_away"].fillna(0)
fixture_features["form_gf_avg_diff"] = fixture_features["form_gf_avg_diff_home"].fillna(0) - fixture_features["form_gf_avg_diff_away"].fillna(0)
fixture_features["form_ga_avg_diff"] = fixture_features["form_ga_avg_diff_home"].fillna(0) - fixture_features["form_ga_avg_diff_away"].fillna(0)

X_fixture = fixture_features[feature_cols]

model = joblib.load(MODEL_PATH)
proba = model.predict_proba(X_fixture)

proba_df = proba_base.copy()
proba_df["proba_0"] = proba[:, 0].astype(float)
proba_df["proba_1"] = proba[:, 1].astype(float)
proba_df["proba_2"] = proba[:, 2].astype(float)

print(f"Matriz de probabilidades: {proba_df.shape[0]} partidos posibles")
proba_df.head()

## 3. Simulación Monte Carlo con grupos fijos

In [ ]:
# Ejecutar simulación Monte Carlo con los grupos oficiales
results, winners_list, advancement = run_fixed_group_tournament(
    groups=GROUPS_2026,
    proba_df=proba_df,
    n_simulations=N_SIMULATIONS,
    random_seed=42,
)

print(f"Simulaciones completadas: {N_SIMULATIONS:,}")
print(f"Equipos que ganaron al menos 1 vez: {len(results)}")

## 4. Top 10 con intervalos de confianza al 95%

Se calculan dos tipos de intervalo de confianza:
- **Wilson Score**: intervalo analítico para proporciones binomiales
- **Bootstrap**: intervalo no paramétrico por re-muestreo

In [ ]:
# Calcular intervalos de confianza (Wilson Score + Bootstrap)
ci_df = compute_all_confidence_intervals(
    winners=winners_list,
    n_simulations=N_SIMULATIONS,
    confidence=0.95,
    n_bootstrap=10_000,
    random_seed=42,
)

# Mostrar Top 10 con intervalos de confianza
top10 = ci_df.head(10).copy()
top10["probability_pct"] = (top10["probability"] * 100).round(2)
top10["wilson_ci"] = top10.apply(
    lambda r: f"[{r['ci_lower_wilson']*100:.2f}%, {r['ci_upper_wilson']*100:.2f}%]",
    axis=1,
)
top10["bootstrap_ci"] = top10.apply(
    lambda r: f"[{r['ci_lower_bootstrap']*100:.2f}%, {r['ci_upper_bootstrap']*100:.2f}%]",
    axis=1,
)

display_cols = ["team", "championships", "probability_pct", "wilson_ci", "bootstrap_ci"]
print("\n" + "="*80)
print("  TOP 10 SELECCIONES — Probabilidad de campeonar (IC 95%)")
print("="*80)
top10[display_cols].rename(columns={
    "team": "Selección",
    "championships": "Campeonatos",
    "probability_pct": "Prob (%)",
    "wilson_ci": "IC Wilson 95%",
    "bootstrap_ci": "IC Bootstrap 95%",
})

In [ ]:
# Visualización: Top 10 con barras de error (IC Wilson)
fig, ax = plt.subplots(figsize=(12, 6))

top10_plot = ci_df.head(10).copy()
teams_plot = top10_plot["team"].values
probs = top10_plot["probability"].values * 100
err_lower = (top10_plot["probability"] - top10_plot["ci_lower_wilson"]).values * 100
err_upper = (top10_plot["ci_upper_wilson"] - top10_plot["probability"]).values * 100

colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(teams_plot)))

bars = ax.barh(
    range(len(teams_plot)),
    probs,
    xerr=[err_lower, err_upper],
    capsize=4,
    color=colors,
    edgecolor="black",
    linewidth=0.5,
    error_kw={"elinewidth": 1.5, "capthick": 1.5, "color": "black"},
)

ax.set_yticks(range(len(teams_plot)))
ax.set_yticklabels(teams_plot, fontsize=11)
ax.invert_yaxis()
ax.set_xlabel("Probabilidad de campeonar (%)", fontsize=12)
ax.set_title(
    f"Top 10 — Probabilidad de ganar el Mundial 2026\n"
    f"({N_SIMULATIONS:,} simulaciones Monte Carlo, IC Wilson 95%)",
    fontsize=13,
    fontweight="bold",
)

# Agregar etiquetas de porcentaje
for i, (p, lo, hi) in enumerate(zip(probs, err_lower, err_upper)):
    ax.text(p + hi + 0.3, i, f"{p:.1f}%", va="center", fontsize=10)

ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "reports" / "figures" / "top10_ci_wilson.png", dpi=150)
plt.show()
print("Figura guardada en reports/figures/top10_ci_wilson.png")

## 5. Predicción de clasificados por grupo (Top 2)

Para cada grupo oficial, se muestran los 2 equipos con mayor probabilidad de avanzar a la fase eliminatoria.

In [ ]:
# Construir tabla de avance por grupo
adv_rows = []
for team, stats in advancement.items():
    adv_rows.append({
        "Grupo": stats["group"],
        "Selección": team,
        "Veces 1ro": stats["first"],
        "Veces 2do": stats["second"],
        "Veces 3ro": stats["third"],
        "Clasificado (top 2)": stats["qualified"],
        "Prob. clasificar (%)": round(stats["qualified"] / N_SIMULATIONS * 100, 2),
    })

adv_df = pd.DataFrame(adv_rows)
adv_df = adv_df.sort_values(["Grupo", "Prob. clasificar (%)"], ascending=[True, False])

print("="*80)
print("  PREDICCIÓN DE CLASIFICADOS POR GRUPO")
print(f"  Basado en {N_SIMULATIONS:,} simulaciones Monte Carlo")
print("="*80)

for group_label in sorted(GROUPS_2026.keys()):
    group_data = adv_df[adv_df["Grupo"] == group_label].reset_index(drop=True)
    print(f"\n--- Grupo {group_label} ---")
    for _, row in group_data.iterrows():
        marker = " ★" if row["Prob. clasificar (%)"] >= 50 else ""
        print(f"  {row['Selección']:<25s}  Clasifica: {row['Prob. clasificar (%)']:>6.1f}%  "
              f"(1ro: {row['Veces 1ro']:>5d}, 2do: {row['Veces 2do']:>5d}, 3ro: {row['Veces 3ro']:>5d}){marker}")

In [ ]:
# Tabla resumen: 2 clasificados por grupo (mayor probabilidad)
print("\n" + "="*60)
print("  SELECCIÓN FINAL: 2 CLASIFICADOS POR GRUPO")
print("="*60)

clasificados_rows = []
for group_label in sorted(GROUPS_2026.keys()):
    group_data = adv_df[adv_df["Grupo"] == group_label].nlargest(2, "Prob. clasificar (%)")
    for rank, (_, row) in enumerate(group_data.iterrows(), 1):
        clasificados_rows.append({
            "Grupo": group_label,
            "Posición": f"{rank}°",
            "Selección": row["Selección"],
            "Prob. clasificar (%)": row["Prob. clasificar (%)"],
        })
        print(f"  Grupo {group_label} — {rank}°: {row['Selección']:<25s} ({row['Prob. clasificar (%)']:.1f}%)")

clasificados_df = pd.DataFrame(clasificados_rows)

# Verificar que no hay repeticiones
selecciones_unicas = clasificados_df["Selección"].nunique()
print(f"\nTotal clasificados: {len(clasificados_df)} (selecciones únicas: {selecciones_unicas})")
clasificados_df

## 6. Validación retrospectiva (Mundiales 2018 y 2022)

Se simulan los mundiales pasados con el mismo modelo para validar la capacidad predictiva.

In [ ]:
# Ejecutar validación retrospectiva
retro_results = run_retrospective_validation(
    proba_df=proba_df,
    n_simulations=N_SIMULATIONS,
    random_seed=42,
)

# Mostrar reporte formateado
report = format_retrospective_report(retro_results)
print(report)

In [ ]:
# Visualización: Validación retrospectiva — Top 10 predicciones vs Resultado real
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, (year, data) in enumerate(sorted(retro_results.items())):
    ax = axes[idx]
    top10_retro = data["top10"][:10]
    teams_r = [t for t, _ in top10_retro]
    counts_r = [c for _, c in top10_retro]
    total = sum(data["predictions"].values())
    probs_r = [c / total * 100 for c in counts_r]

    colors_r = [
        "#FFD700" if t == data["actual_winner"]
        else "#C0C0C0" if t in data["actual_top4"]
        else "#4A90D9"
        for t in teams_r
    ]

    ax.barh(range(len(teams_r)), probs_r, color=colors_r, edgecolor="black", linewidth=0.5)
    ax.set_yticks(range(len(teams_r)))
    ax.set_yticklabels(teams_r, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel("Probabilidad (%)", fontsize=11)
    ax.set_title(
        f"Mundial {year} — Predicción vs Realidad\n"
        f"Ganador real: {data['actual_winner']} (predicho #{data['winner_predicted_rank']})",
        fontsize=11,
        fontweight="bold",
    )

    for i, p in enumerate(probs_r):
        ax.text(p + 0.2, i, f"{p:.1f}%", va="center", fontsize=9)

    ax.grid(axis="x", alpha=0.3)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#FFD700", edgecolor="black", label="Campeón real"),
    Patch(facecolor="#C0C0C0", edgecolor="black", label="Top 4 real"),
    Patch(facecolor="#4A90D9", edgecolor="black", label="Otro"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3, fontsize=10)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig(PROJECT_ROOT / "reports" / "figures" / "retrospective_validation.png", dpi=150)
plt.show()
print("Figura guardada en reports/figures/retrospective_validation.png")

## 7. Tabla completa de probabilidades (todas las selecciones)

In [ ]:
# Tabla completa con IC para todas las selecciones
full_table = ci_df.copy()
full_table["probability_pct"] = (full_table["probability"] * 100).round(3)
full_table["ci_lower_wilson_pct"] = (full_table["ci_lower_wilson"] * 100).round(3)
full_table["ci_upper_wilson_pct"] = (full_table["ci_upper_wilson"] * 100).round(3)
full_table["ci_lower_bootstrap_pct"] = (full_table["ci_lower_bootstrap"] * 100).round(3)
full_table["ci_upper_bootstrap_pct"] = (full_table["ci_upper_bootstrap"] * 100).round(3)

display_full = full_table[[
    "team", "championships", "probability_pct",
    "ci_lower_wilson_pct", "ci_upper_wilson_pct",
    "ci_lower_bootstrap_pct", "ci_upper_bootstrap_pct",
]].rename(columns={
    "team": "Selección",
    "championships": "Campeonatos",
    "probability_pct": "Prob (%)",
    "ci_lower_wilson_pct": "IC Wilson Inf (%)",
    "ci_upper_wilson_pct": "IC Wilson Sup (%)",
    "ci_lower_bootstrap_pct": "IC Boot Inf (%)",
    "ci_upper_bootstrap_pct": "IC Boot Sup (%)",
})

display_full

## Notas
- Los **grupos oficiales** del sorteo FIFA 2026 se usan en todas las simulaciones.
- Los **intervalos de confianza** se calculan con Wilson Score (analítico) y Bootstrap (10,000 re-muestras).
- La **predicción de clasificados** por grupo se basa en la frecuencia de avanzar a la siguiente fase en las simulaciones.
- La **validación retrospectiva** simula los mundiales 2018 y 2022 con el mismo modelo para evaluar su poder predictivo.
- `home_advantage = 0` para todos los partidos del mundial (cancha neutral).